# Série Histórica de Preços — Perfil Exploratório

Análise das séries de preços de combustíveis coletados pela ANP (LPC).

**Dados:** dsan gasolina/etanol (1,17M), dsan diesel/GNV (537k), LPC consolidado (1,24M)  
**Período:** 2024-01 a 2026-05  
**Trusted:** `data/trusted/serie-historica-precos/`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

REPO = Path.cwd().parents[2]
TRUSTED = REPO / 'data' / 'trusted' / 'serie-historica-precos'

gas_etanol = pd.read_parquet(TRUSTED / 'dsan_gasolina_etanol_2024_2025.parquet')
diesel_gnv = pd.read_parquet(TRUSTED / 'dsan_diesel_gnv_2024_2025.parquet')

# Consolidar todos os produtos em um único df
df = pd.concat([gas_etanol, diesel_gnv], ignore_index=True)
df['data_coleta'] = pd.to_datetime(df['data_coleta'])
df['mes'] = df['data_coleta'].dt.to_period('M')

print(f'Total: {len(df):,} coletas')
print(f'CNPJs: {df.cnpj.nunique():,}')
print(f'Período: {df.data_coleta.min().date()} a {df.data_coleta.max().date()}')
print(f'\nProdutos:')
print(df['produto'].value_counts().to_string())

## 1. Evolução do preço médio mensal por produto

In [ ]:
preco_mensal = df.groupby(['mes', 'produto'])['valor_venda'].mean().unstack()

fig, ax = plt.subplots(figsize=(14, 5))
for col in preco_mensal.columns:
    ax.plot(preco_mensal.index.to_timestamp(), preco_mensal[col], label=col, linewidth=1.2)

ax.set_title('Preço médio de venda — evolução mensal (2024–2025)')
ax.set_ylabel('R$ / litro (ou m³ para GNV)')
ax.legend(fontsize=8, loc='upper left')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Spread compra/venda — margem do revendedor

In [ ]:
# Apenas registros com ambos valores
com_compra = df[df['valor_compra'].notna() & (df['valor_compra'] > 0)].copy()
com_compra['spread'] = com_compra['valor_venda'] - com_compra['valor_compra']

print(f'Registros com valor_compra: {len(com_compra):,} ({len(com_compra)/len(df)*100:.1f}%)')
print(f'\nSpread médio (R$/L) por produto:')
spread_prod = com_compra.groupby('produto')['spread'].agg(['mean', 'median', 'count'])
print(spread_prod.round(3).to_string())

# Evolução do spread mensal - top 3 produtos
top3 = com_compra['produto'].value_counts().head(3).index.tolist()
spread_mensal = com_compra[com_compra['produto'].isin(top3)].groupby(
    ['mes', 'produto'])['spread'].median().unstack()

fig, ax = plt.subplots(figsize=(14, 4))
for col in spread_mensal.columns:
    ax.plot(spread_mensal.index.to_timestamp(), spread_mensal[col], label=col, linewidth=1.2)
ax.set_title('Spread mediano (venda − compra) — evolução mensal')
ax.set_ylabel('R$ / litro')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Distribuição geográfica — preço médio por UF (gasolina)

In [ ]:
gasolina = df[df['produto'] == 'GASOLINA'].copy()

preco_uf = gasolina.groupby('uf')['valor_venda'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(preco_uf.index, preco_uf['mean'], color='steelblue', alpha=0.7, label='Média')
ax.scatter(preco_uf.index, preco_uf['median'], color='red', s=20, zorder=5, label='Mediana')
ax.axhline(gasolina['valor_venda'].mean(), ls='--', color='gray', label=f'Nacional: R$ {gasolina["valor_venda"].mean():.2f}')
ax.set_title('Preço médio Gasolina por UF (2024–2025)')
ax.set_ylabel('R$ / litro')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 4. Diesel S10 vs Diesel comum — gap regional

In [ ]:
diesel = df[df['produto'].isin(['DIESEL', 'DIESEL S10'])].copy()

diesel_uf = diesel.groupby(['uf', 'produto'])['valor_venda'].mean().unstack(fill_value=0)
if 'DIESEL' in diesel_uf.columns and 'DIESEL S10' in diesel_uf.columns:
    diesel_uf['gap'] = diesel_uf['DIESEL S10'] - diesel_uf['DIESEL']
    diesel_uf = diesel_uf.sort_values('gap', ascending=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(diesel_uf))
    ax.bar(x, diesel_uf['DIESEL'], label='Diesel comum', color='gray', alpha=0.7)
    ax.bar(x, diesel_uf['gap'], bottom=diesel_uf['DIESEL'], label='Premium S10', color='orange', alpha=0.7)
    ax.set_xticks(x)
    ax.set_xticklabels(diesel_uf.index, rotation=45)
    ax.set_title('Diesel comum + gap S10 por UF')
    ax.set_ylabel('R$ / litro')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f'\nGap médio S10-Diesel: R$ {diesel_uf["gap"].mean():.3f}/L')
    print(f'UF com maior gap: {diesel_uf["gap"].idxmax()} (R$ {diesel_uf["gap"].max():.3f})')
    print(f'UF com menor gap: {diesel_uf["gap"].idxmin()} (R$ {diesel_uf["gap"].min():.3f})')

## 5. Preço por bandeira — gasolina (boxplot top 10)

In [ ]:
top_band = gasolina['bandeira'].value_counts().head(10).index.tolist()
gas_band = gasolina[gasolina['bandeira'].isin(top_band)]

fig, ax = plt.subplots(figsize=(12, 5))
data_box = [gas_band[gas_band['bandeira'] == b]['valor_venda'].dropna().values for b in top_band]
bp = ax.boxplot(data_box, labels=top_band, patch_artist=True, showfliers=False)
for patch in bp['boxes']:
    patch.set_facecolor('lightskyblue')
ax.set_title('Distribuição preço Gasolina — Top 10 bandeiras')
ax.set_ylabel('R$ / litro')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

# Tabela resumo
resumo = gas_band.groupby('bandeira')['valor_venda'].agg(['mean','median','std','count']).sort_values('mean')
print(resumo.round(3).to_string())

## 6. Variação semanal — volatilidade por região

In [ ]:
gasolina['semana'] = gasolina['data_coleta'].dt.isocalendar().week.astype(int)
gasolina['ano'] = gasolina['data_coleta'].dt.year

# Preço médio semanal por região
semanal_regiao = gasolina.groupby(['ano', 'semana', 'regiao'])['valor_venda'].mean().reset_index()

# Variação percentual semana a semana
vol_regiao = semanal_regiao.groupby('regiao')['valor_venda'].agg(
    lambda x: x.pct_change().std() * 100
).sort_values(ascending=False)

print('Volatilidade semanal (% std) — Gasolina por região:')
print(vol_regiao.round(3).to_string())

# Série semanal
fig, ax = plt.subplots(figsize=(14, 4))
for regiao in semanal_regiao['regiao'].unique():
    subset = semanal_regiao[semanal_regiao['regiao'] == regiao].sort_values(['ano', 'semana'])
    ax.plot(range(len(subset)), subset['valor_venda'].values, label=regiao, linewidth=0.8)
ax.set_title('Preço médio semanal Gasolina por região')
ax.set_ylabel('R$ / litro')
ax.set_xlabel('Semanas (2024–2025)')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. GNV — cobertura e preço

In [ ]:
gnv = df[df['produto'] == 'GNV'].copy()
print(f'GNV: {len(gnv):,} coletas, {gnv.cnpj.nunique():,} postos')
print(f'UFs com GNV: {gnv.uf.nunique()} — {sorted(gnv.uf.unique())}')
print(f'Preço médio: R$ {gnv.valor_venda.mean():.3f}/m³')
print(f'\nPor UF (top 10 por volume de coletas):')

gnv_uf = gnv.groupby('uf').agg(
    coletas=('valor_venda', 'count'),
    preco_medio=('valor_venda', 'mean'),
    postos=('cnpj', 'nunique')
).sort_values('coletas', ascending=False)
print(gnv_uf.head(10).round(3).to_string())

## 8. Etanol/Gasolina — paridade (70%)

In [ ]:
# Paridade etanol/gasolina por UF-mês
gas_mes_uf = df[df['produto'] == 'GASOLINA'].groupby(['mes', 'uf'])['valor_venda'].mean()
eta_mes_uf = df[df['produto'] == 'ETANOL'].groupby(['mes', 'uf'])['valor_venda'].mean()

paridade = (eta_mes_uf / gas_mes_uf).reset_index(name='razao')
paridade['vantagem_etanol'] = paridade['razao'] < 0.7

print(f'Observações UF-mês: {len(paridade)}')
print(f'Etanol vantajoso (<70%): {paridade.vantagem_etanol.sum()} ({paridade.vantagem_etanol.mean()*100:.1f}%)')

# Por UF: % meses com vantagem
vant_uf = paridade.groupby('uf')['vantagem_etanol'].mean().sort_values(ascending=False) * 100

fig, ax = plt.subplots(figsize=(12, 5))
vant_uf.plot(kind='bar', ax=ax, color='green', alpha=0.7)
ax.axhline(50, ls='--', color='red', label='50%')
ax.set_title('% meses com etanol vantajoso (razão < 70%) por UF — 2024–2025')
ax.set_ylabel('% meses')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 9. Dispersão de preços — municípios mais caros vs mais baratos

In [ ]:
# Gasolina — municípios com >= 50 coletas
mun_gas = gasolina.groupby(['municipio', 'uf']).agg(
    preco_medio=('valor_venda', 'mean'),
    coletas=('valor_venda', 'count')
).reset_index()
mun_gas = mun_gas[mun_gas['coletas'] >= 50]

print(f'Municípios com >= 50 coletas: {len(mun_gas)}')
print(f'\nTop 10 MAIS CAROS:')
caros = mun_gas.nlargest(10, 'preco_medio')
for _, r in caros.iterrows():
    print(f"  {r['municipio']:25s} {r['uf']}  R$ {r['preco_medio']:.3f}  ({int(r['coletas'])} coletas)")

print(f'\nTop 10 MAIS BARATOS:')
baratos = mun_gas.nsmallest(10, 'preco_medio')
for _, r in baratos.iterrows():
    print(f"  {r['municipio']:25s} {r['uf']}  R$ {r['preco_medio']:.3f}  ({int(r['coletas'])} coletas)")

print(f'\nAmplitude: R$ {mun_gas.preco_medio.max() - mun_gas.preco_medio.min():.3f}/L')
print(f'CV nacional: {mun_gas.preco_medio.std() / mun_gas.preco_medio.mean() * 100:.2f}%')

## 10. Resumo estatístico

In [ ]:
resumo = df.groupby('produto').agg(
    coletas=('valor_venda', 'count'),
    postos=('cnpj', 'nunique'),
    preco_medio=('valor_venda', 'mean'),
    preco_mediano=('valor_venda', 'median'),
    preco_min=('valor_venda', 'min'),
    preco_max=('valor_venda', 'max'),
    cv_pct=('valor_venda', lambda x: x.std()/x.mean()*100)
).sort_values('coletas', ascending=False)

print('Resumo por produto (2024–2025):')
print(resumo.round(3).to_string())
print(f'\nTotal geral: {len(df):,} coletas em {df.cnpj.nunique():,} postos')